In [ ]:
import pandas as pd
import json
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
import joblib

print("Încărcăm dataset-ul...")
df = pd.read_json('dataset_v6_augmented.json')

df['text'] = df['text'].str.lower()

print(f"Total instanțe: {len(df)}")
print(df['category'].value_counts())
print("-" * 50)

X_train, X_test, y_train, y_test = train_test_split(
    df['text'],
    df['category'],
    test_size=0.2,
    random_state=42,
    stratify=df['category']
)

# Folosim TF-IDF pentru a vectoriza cuvintele și n-gramele (grupuri de 1 sau 2 cuvinte)
# Folosim LinearSVC (Support Vector Classifier), excelent pentru clasificarea textului
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_df=0.9, min_df=2)),
    ('clf', LinearSVC(class_weight='balanced', random_state=42))
])

print("Antrenăm modelul...")
pipeline.fit(X_train, y_train)

nume_fisier = 'model_hate_speech.pkl'
joblib.dump(pipeline, nume_fisier)
print(f"Modelul a fost salvat cu succes ca: {nume_fisier}")

print("Evaluăm modelul pe datele de test...")
y_pred = pipeline.predict(X_test)
print("\nRaport de Clasificare:\n")
print(classification_report(y_test, y_pred))

def prezice_hate_speech(text):
    text_curat = text.lower()
    predictie = pipeline.predict([text_curat])[0]
    return f"Text: '{text}' \nCategorie detectată: {predictie}\n"

print("-" * 50)
print("Testează modelul cu propriile propoziții:")
print(prezice_hate_speech("Să vă dau la toți din guvern că ați distrus țara asta!"))
print(prezice_hate_speech("Arbitrule ești complet orb, un ratat!"))
print(prezice_hate_speech("Femeia la cratiță, nu are ce căuta în funcții de conducere."))

In [ ]:
import joblib

model_incarcat = joblib.load('model_hate_speech.pkl')
print("Model încărcat cu succes!")

text_nou = "ce pasarica ai in tine fata mea"
predictie = model_incarcat.predict([text_nou])[0]

print(f"Text: '{text_nou}'")
print(f"Categorie detectată: {predictie}")

In [ ]:
!pip install transformers datasets evaluate accelerate scikit-learn

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate

df = pd.read_json('categorized_reddit_data_filtrat.json')

# Modelele au nevoie de etichete numerice (0, 1, 2...), nu text
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['category'])

# Salvăm maparea claselor (pentru a ști mai târziu ce înseamnă 0, 1 etc.)
id2label = {i: label for i, label in enumerate(label_encoder.classes_)}
label2id = {label: i for i, label in enumerate(label_encoder.classes_)}
num_labels = len(id2label)

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Tokenizarea (Transformarea textului pentru RoBERTa)
nume_model = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(nume_model)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

print("Tokenizăm textul...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# Configurarea Modelului RoBERTa
print("Descărcăm modelul XLM-RoBERTa...")
model = AutoModelForSequenceClassification.from_pretrained(
    nume_model,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

# Parametrii de antrenare
training_args = TrainingArguments(
    output_dir="./roberta_hate_speech",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
)

# Metrica pentru evaluare
metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# Inițializarea și rularea Trainer-ului
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print("Începem antrenamentul")
trainer.train()

# Salvarea finală a modelulu
cale_salvare = "./roberta_hate_speech_final"
trainer.save_model(cale_salvare)
tokenizer.save_pretrained(cale_salvare)
print(f"Modelul complet a fost salvat în folderul: {cale_salvare}")

In [ ]:
import pandas as pd
from transformers import pipeline

cale_model = "./roberta_hate_speech_final"

clasificator_roberta = pipeline("text-classification", model=cale_model, tokenizer=cale_model)
df_test = pd.read_json('injuraturi_test_json.json')

if 'text' not in df_test.columns:
    print("Eroare: Fișierul JSON nu pare să aibă o coloană numită 'text'. Verifică structura fișierului.")
else:

    texte_de_testat = df_test['text'].tolist()

    print(f"Generăm predicțiile pentru {len(texte_de_testat)} de propoziții")
    rezultate = clasificator_roberta(texte_de_testat)

    df_test['Categorie_Prezisa'] = [rez['label'] for rez in rezultate]
    df_test['Scor_Incredere'] = [rez['score'] for rez in rezultate]

    print("\n" + "="*60)
    print("  PREVIZUALIZARE REZULTATE (Primele 10 propoziții)")
    print("="*60)

    for index, row in df_test.head(10).iterrows():
        print(f"Text: '{row['text']}'")
        print(f"Predicție: {row['Categorie_Prezisa']} | Încredere: {row['Scor_Incredere']:.4f}")
        # Dacă JSON-ul de test are deja categoria reală, o afișăm
        if 'category' in row:
            print(f"Categorie reală (din JSON): {row['category']}")
        print("-" * 60)

    nume_fisier_salvat = 'rezultate_predictii_test.json'
    df_test.to_json(nume_fisier_salvat, orient='records', force_ascii=False, indent=4)
    print(f"\nToate predicțiile au fost salvate în fișierul '{nume_fisier_salvat}'.")

In [ ]:
!zip -r model_roberta.zip ./roberta_hate_speech_final

In [ ]:
import pandas as pd
from google.colab import files

print("Te rog încarcă fișierul categorized_reddit_data.json:")
uploaded = files.upload()

# 2. Citește datele într-un DataFrame Pandas
nume_fisier = 'categorized_reddit_data.json'
df = pd.read_json(nume_fisier)

print(f"Număr total de intrări înainte de filtrare: {len(df)}")

# 3. Filtrează datele - păstrăm tot ce NU este "Online Forums & Social Media"
# Folosim != pentru a exclude categoria dorită
df_filtrat = df[df['category'] != "Online Forums & Social Media"]

print(f"Număr de intrări după eliminarea categoriei: {len(df_filtrat)}")

# 4. Salvează rezultatul într-un nou fișier JSON
nume_fisier_nou = 'categorized_reddit_data_filtrat.json'
df_filtrat.to_json(nume_fisier_nou, orient='records', force_ascii=False, indent=4)

print(f"Fișierul a fost salvat ca: {nume_fisier_nou}")

# 5. Descarcă automat fișierul curățat pe calculatorul tău
files.download(nume_fisier_nou)